# 05 - Modèle final embarqué et décision « je ne sais pas »

Modèle retenu : **petit U-Net, Sentinel-2 seul, 60 époques** (expérience B du notebook 04).
Il ne dépend que de l'image optique : il peut tourner à bord d'un seul satellite et sur n'importe
quelle image Sentinel-2 fournie pour la démonstration.

1. **Entraîner** le modèle final.
2. **L'exporter** au format ONNX (standard, lisible hors de PyTorch) puis le **compresser en 8 bits**.
3. **Vérifier** que la version compressée donne les mêmes résultats.
4. **Savoir dire « je ne sais pas »** : décision à quatre issues (alerte, pas d'alerte, incertain, inconnu).
5. **Tester l'inconnu** sur des images volontairement anormales.

Exécution : Colab avec GPU T4, *Exécution > Tout exécuter* (environ 20 minutes).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "onnx", "onnxruntime"], check=True)

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from src import donnees, modeles, entrainement, alerte, embarque, decision
from src.metriques import scores_alerte

noms = donnees.telecharger("data")
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
print(len(noms), "images | calcul sur", APPAREIL, torch.cuda.get_device_name(0) if APPAREIL == "cuda" else "")
if APPAREIL == "cpu":
    print("ATTENTION : pas de GPU. Exécution > Modifier le type d'exécution > GPU T4, puis relancer.")

## 1. Données

In [ ]:
images, labels, perms = [], [], []
for nom in noms:
    s2, label, perm = donnees.lire_image(nom, "data")
    images.append(s2.astype(np.int16))
    labels.append(label.astype(np.int8))
    perms.append(perm)

def s2f(k):
    """Image k en nombres à virgule, calculée à la demande (économise la mémoire)."""
    return images[k].astype(np.float32)

df = pd.DataFrame({"image": noms, "pays": [donnees.pays(n) for n in noms], "partie": [donnees.partie(n) for n in noms]})
df["part_inondee"] = [donnees.part_inondee_vraie(l.astype(np.float32), p) for l, p in zip(labels, perms)]
df["nuages_annotes"] = [float((l < 0).mean()) for l in labels]
SEUIL_ALERTE = 0.05
df["alerte_vraie"] = df["part_inondee"] >= SEUIL_ALERTE

idx = {p: df.index[df["partie"] == p].tolist() for p in ["train", "validation", "test"]}
train = (df["partie"] == "train").to_numpy()
moyenne, ecart = entrainement.statistiques_bandes(images, idx["train"])
print({p: len(i) for p, i in idx.items()})

## 2. Entraînement du modèle final (petit U-Net, Sentinel-2, 60 époques)

In [ ]:
EPOQUES = 60 if APPAREIL == "cuda" else 2
os.makedirs("modeles", exist_ok=True)
chemin = f"modeles/petit_unet_s2_{EPOQUES}ep"
modele = modeles.Normalise(modeles.PetitUNet(canaux=13), moyenne, ecart)
if os.path.exists(chemin + ".pt"):          # reprise si la session a déjà entraîné ce modèle
    modele.load_state_dict(torch.load(chemin + ".pt", map_location="cpu"))
    modele.to(APPAREIL)
    historique = pd.read_csv(chemin + ".csv") if os.path.exists(chemin + ".csv") else None
    print("déjà entraîné : poids rechargés")
else:
    historique = entrainement.entrainer(modele, images, labels, idx["train"], idx["validation"],
                                        APPAREIL, epoques=EPOQUES)
    torch.save(modele.state_dict(), chemin + ".pt")
    historique.to_csv(chemin + ".csv", index=False)
if historique is not None:
    print(f"meilleure IoU de validation {historique['iou_validation'].max():.3f} "
          f"(époque {int(historique.loc[historique['iou_validation'].idxmax(), 'epoque'])})")

## 3. Export ONNX et compression en 8 bits

- **ONNX** : format standard, exécutable sans PyTorch par ONNX Runtime (disponible sur processeurs
  embarqués ARM). La normalisation des bandes est incluse : le modèle prend les réflectances brutes.
- **8 bits** : les poids et les calculs des convolutions passent de nombres à virgule sur 32 bits à
  des entiers sur 8 bits. La plage des valeurs internes est mesurée sur 16 images d'entraînement.

In [ ]:
FP32, INT8 = "modeles/one_look_fp32.onnx", "modeles/one_look_int8.onnx"
embarque.exporter_onnx(modele, FP32)
calibration = [images[k] for k in idx["train"][::max(1, len(idx["train"]) // 16)][:16]]
embarque.quantifier_int8(FP32, INT8, calibration)
modele.to(APPAREIL)

sessions = {"ONNX FP32": embarque.session(FP32, fils=2), "ONNX INT8": embarque.session(INT8, fils=2)}
k = idx["validation"][0]
with torch.no_grad():
    p_torch = torch.sigmoid(modele.cpu()(torch.from_numpy(s2f(k)[None]))).numpy()[0, 0]
modele.to(APPAREIL)
for nom, sess in sessions.items():
    print(f"{nom} : écart maximal avec PyTorch sur une image = {np.abs(embarque.probabilite_eau(sess, s2f(k)) - p_torch).max():.5f}")
print(f"Taille : FP32 {embarque.taille_fichier_mo(FP32):.2f} Mo | INT8 {embarque.taille_fichier_mo(INT8):.2f} Mo")

## 4. La version compressée donne-t-elle les mêmes résultats ?

Même protocole que d'habitude : seuil `t_alerte` choisi sur l'entraînement pour chaque version,
scores sur la validation et le test. Temps mesuré sur **un seul cœur** de processeur.

In [ ]:
tous = df.index.tolist()
probs = {"PyTorch": entrainement.predire(modele, images, tous, APPAREIL)}
for nom, sess in sessions.items():
    debut = time.time()
    probs[nom] = embarque.predire(sess, images, tous)
    print(f"{nom} : {len(tous)} images en {time.time() - debut:.0f} s")

parts, seuils_alerte, lignes = {}, {}, []
for nom, p in probs.items():
    parts[nom] = np.array([alerte.part_eau_reseau(p[k], s2f(k), perms[k]) for k in tous])
    seuils_alerte[nom], _ = alerte.choisir_seuil_alerte(df["alerte_vraie"][train], parts[nom][train], SEUIL_ALERTE)
    ligne = {"version": nom, "t_alerte": seuils_alerte[nom]}
    if nom == "PyTorch":
        ligne.update(taille_Mo=round(modeles.taille_mo(modele), 2),
                     ms_1_coeur=round(entrainement.latence_cpu_ms(modele), 0))
    else:
        f = FP32 if nom == "ONNX FP32" else INT8
        ligne.update(taille_Mo=round(embarque.taille_fichier_mo(f), 2),
                     ms_1_coeur=round(embarque.latence_ms(embarque.session(f, fils=1)), 0))
    for part in ["validation", "test"]:
        sel = (df["partie"] == part).to_numpy()
        s = scores_alerte(df["alerte_vraie"][sel], parts[nom][sel] >= seuils_alerte[nom])
        ligne.update({f"iou_{part}": round(entrainement.iou_eau(p, labels, idx[part]), 3),
                      f"f1_{part}": s["f1"], f"fausses_alertes_{part}": s["taux_fausse_alerte"]})
    lignes.append(ligne)
comparaison = pd.DataFrame(lignes).set_index("version")
print(comparaison.to_string())
os.makedirs("resultats", exist_ok=True)
comparaison.to_csv("resultats/versions_05.csv")

## 5. Savoir dire « je ne sais pas »

Le modèle embarqué est la version **INT8**. Pour chaque image, trois indicateurs calculés à bord,
sans aucune annotation :

- **nuages** : part des pixels très brillants dans le bleu (bande B2 > 0,2), typique des nuages ;
- **ambiguïté** : part des pixels où le réseau hésite (probabilité d'eau entre 0,2 et 0,8) ;
- **score d'inconnu** : écart de l'image à tout ce qui a été vu à l'entraînement.

Décision (dans cet ordre) :
1. score d'inconnu trop élevé → **INCONNU** (aucune conclusion possible) ;
2. assez d'eau anormale visible → **ALERTE**, même si une partie de l'image est cachée ;
3. trop de nuages (≥ 30 %) ou trop d'ambiguïté → **INCERTAIN** ;
4. sinon → **PAS D'ALERTE**.

Les seuils d'ambiguïté et d'inconnu sont fixés sur l'entraînement uniquement.

In [ ]:
P = probs["ONNX INT8"]
df["part_eau"] = parts["ONNX INT8"]
df["nuages"] = [decision.part_nuages(s2f(k)) for k in tous]
df["ambiguite"] = [decision.ambiguite(P[k], s2f(k), perms[k]) for k in tous]
detecteur = decision.DetecteurInconnu([decision.signature(s2f(k)) for k in idx["train"]])
df["score_inconnu"] = [detecteur.score(s2f(k)) for k in tous]

r = np.corrcoef(df.loc[train, "nuages"], df.loc[train, "nuages_annotes"])[0, 1]
print(f"Nuages estimés à bord vs zones non annotées (entraînement) : corrélation {r:.2f}")

SEUILS = {"alerte": seuils_alerte["ONNX INT8"], "nuages": 0.30,
          "ambiguite": round(float(df.loc[train, "ambiguite"].quantile(0.95)), 4),
          "inconnu": round(detecteur.seuil, 3)}
print("Seuils :", SEUILS)

def decisions_pour(seuils, sel):
    return [decision.decider(r.part_eau, r.nuages, r.ambiguite, r.score_inconnu, seuils) for r in df[sel].itertuples()]

lignes = []
for part in ["validation", "test"]:
    sel = (df["partie"] == part).to_numpy()
    deux = ["ALERTE" if e >= SEUILS["alerte"] else "PAS D'ALERTE" for e in df.loc[sel, "part_eau"]]
    lignes.append({"partie": part, "decision": "2 issues (oui / non)", **decision.bilan(deux, df.loc[sel, "alerte_vraie"])})
    lignes.append({"partie": part, "decision": "4 issues", **decision.bilan(decisions_pour(SEUILS, sel), df.loc[sel, "alerte_vraie"])})
bilan = pd.DataFrame(lignes).set_index(["partie", "decision"])
print(bilan[["ALERTE", "PAS D'ALERTE", "INCERTAIN", "INCONNU", "couverture", "precision", "rappel",
             "taux_fausse_alerte", "inondations_manquees_sans_prevenir", "inondations_signalees_incertaines"]].to_string())
bilan.to_csv("resultats/decision_05.csv")

### Sensibilité au seuil de nuages

Plus le seuil est bas, plus le système se déclare incertain : il manque moins d'inondations sans
prévenir, mais décide sur moins d'images. Validation et test réunis.

In [ ]:
eval_ = (df["partie"] != "train").to_numpy()
lignes = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 1.01]:
    b = decision.bilan(decisions_pour({**SEUILS, "nuages": t}, eval_), df.loc[eval_, "alerte_vraie"])
    lignes.append({"seuil_nuages": "aucun" if t > 1 else t, **{c: b[c] for c in
                   ["INCERTAIN", "couverture", "f1", "taux_fausse_alerte",
                    "inondations_manquees_sans_prevenir", "inondations_signalees_incertaines"]}})
print(pd.DataFrame(lignes).to_string(index=False))

### Exemples

In [ ]:
def montrer(k):
    s2 = s2f(k)
    d = decision.decider(df.part_eau[k], df.nuages[k], df.ambiguite[k], df.score_inconnu[k], SEUILS)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(donnees.rgb(s2)); ax[0].set_title(f"{noms[k]} → {d}")
    ax[1].imshow((P[k] > 0.5) & donnees.pixels_utiles(s2, perms[k]), cmap="Blues")
    ax[1].set_title(f"Eau détectée {df.part_eau[k]:.1%} | nuages {df.nuages[k]:.0%}")
    ax[2].imshow((labels[k] == 1) & ~perms[k], cmap="Blues"); ax[2].set_title(f"Annoté : {df.part_inondee[k]:.1%}")
    for a in ax: a.axis("off")
    plt.show()

for nom in ["Ghana_866994", "Ghana_313799", "Pakistan_246718", "Pakistan_211386", "Bolivia_103757"]:
    if nom in noms:
        montrer(noms.index(nom))

## 6. Face à l'inconnu

Images volontairement anormales, fabriquées à partir de 3 images de validation :
- **capteur saturé** : toutes les réflectances multipliées par 3 ;
- **bandes perdues** : les bandes infrarouges (B8 à B12) mises à zéro ;
- **neige** (simulation grossière) : très brillant dans le visible et le proche infrarouge,
  sombre dans l'infrarouge moyen (B11, B12), comme la neige réelle ;
- **bruit** : image aléatoire.

Un bon système doit répondre INCONNU plutôt qu'ALERTE ou PAS D'ALERTE.

In [ ]:
sess8 = embarque.session(INT8, fils=2)
def neige(x):
    y = x.copy(); y[:10] = 8000; y[10:] = 800
    return y
anomalies = {
    "originale": lambda x: x,
    "capteur saturé": lambda x: x * 3,
    "bandes perdues": lambda x: np.concatenate([x[:7], np.zeros_like(x[7:])]),
    "neige": neige,
    "bruit": lambda x: np.random.default_rng(0).uniform(0, 10000, x.shape).astype(np.float32),
}
lignes = []
for k in idx["validation"][:3]:
    for nom, f in anomalies.items():
        x = f(s2f(k))
        p = embarque.probabilite_eau(sess8, x)
        d = decision.decider(alerte.part_eau_reseau(p, x, perms[k]), decision.part_nuages(x),
                             decision.ambiguite(p, x, perms[k]), detecteur.score(x), SEUILS)
        lignes.append({"image": noms[k], "transformation": nom, "score_inconnu": round(detecteur.score(x), 1), "decision": d})
print(f"Seuil d'inconnu : {SEUILS['inconnu']}")
print(pd.DataFrame(lignes).to_string(index=False))

## 7. Configuration et fichiers du modèle final

Tout ce qu'il faut pour utiliser le modèle hors de ce notebook (démonstration) : le fichier ONNX
et les seuils de décision.

In [ ]:
config = {
    "modele": "one_look_int8.onnx",
    "entree": "reflectances Sentinel-2, 13 bandes (B1 a B12 et B8A), entiers L1C, forme (1, 13, H, W), H et W multiples de 8",
    "sortie": "logits d'eau, forme (1, 1, H, W) ; probabilite = sigmoide(logits)",
    "seuil_probabilite_eau": 0.5,
    "definition_alerte": f"au moins {SEUIL_ALERTE:.0%} de l'image inondee hors eau permanente",
    "seuils_decision": SEUILS,
    "seuil_bleu_nuage": decision.SEUIL_BLEU_NUAGE,
    "detecteur_inconnu": detecteur.config(),
}
with open("modeles/config.json", "w") as f:
    json.dump(config, f, indent=1, ensure_ascii=False)
print(json.dumps(config, ensure_ascii=False))
for f in sorted(os.listdir("modeles")):
    print(f"{f:32s} {os.path.getsize('modeles/' + f) / 1e6:.2f} Mo")

## À retenir

- Le modèle final tient en **moins d'un mégaoctet** en 8 bits et se lit sans PyTorch.
- La compression ne doit pas dégrader l'IoU ni les alertes : à vérifier dans le tableau de la section 4.
- Dire **INCERTAIN** sous les nuages évite d'affirmer « pas d'inondation » quand le sol n'est pas visible.